# 2. Generating Ground Truth Data

In [ ]:
%load_ext autoreload
%autoreload 2
import dotenv

dotenv.load_dotenv(override=True)

In [ ]:
from src import FaqHttpLoader

loader = FaqHttpLoader()
documents = loader.load()

In [ ]:
print(documents[0]['id'])
print(documents[0]['question'])

Generating questions with structured output

In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.
""".strip()

In [ ]:
from openai import OpenAI

ollama_client = OpenAI(
    api_key='ollama',
    base_url='http://localhost:11434/v1',
)

def llm_structured(
    instructions,
    user_prompt,
    output_type,
    model='granite4.1:8b'
    ):
    messages = [
        {'role': 'system', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = ollama_client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=output_type,
        max_tokens=1024,
    )

    return response.choices[0].message.parsed

In [ ]:
import json

result = llm_structured(
    data_gen_instructions,
    json.dumps(documents[0]),
    Questions
)

print(result.questions)

Parallel processing

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

CACHE_DIR = Path('../../data/ground_truth')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results


def process(doc):
    cache_file = CACHE_DIR / f"{doc['id']}.json"

    if cache_file.exists():
        return json.loads(cache_file.read_text())

    out = llm_structured(
        data_gen_instructions,
        json.dumps(doc),
        Questions
    )

    results = [
        {'question': q, 'course': doc['course'], 'document': doc['id']}
        for q in out.questions
    ]

    cache_file.write_text(json.dumps(results, ensure_ascii=False, indent=2))
    return results

Generate questions for all documents:

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    ground_truth = map_progress(pool, documents[:16], process)

Flatten the nested lists into a single dataset:

In [ ]:
ground_truth